In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
import requests

docs_url = 'https://datatalksclub.github.io/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

In [14]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1470

In [15]:
from minsearch import Index

# SQL query that searches the database. Key word fields are WHERE clauses
index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

# applying the Index to the dataset, executing the query on database
index.fit(documents)


In [10]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [30]:
INSTRUCTIONS = '''
    Your task is to answer questions fromt he course participants based in the provided context.

    Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I dont know."
'''

In [31]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [27]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [18]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()


In [5]:
question = f'I just discovered this course. Can I still join it?'

In [32]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]
    
    response = openai_client.responses.create(
        model = model,
        input = message_history
    )

    return response.output_text

In [33]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(INSTRUCTIONS, user_prompt)

In [34]:
rag(question)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'